# Section 1 Assignment - Transformers

**Student Name:** Stephen White<br>
**Student ID:** 13159178

In [1]:
from huggingface_hub import login, whoami
import os
import transformers
from datasets import load_dataset

/home/stephen-white/.pyenv/versions/3.14.3/envs/mn5162/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 2008

## Product "Letters"

I've used the following from transformers datasets:
https://huggingface.co/datasets/kevykibbz/Amazon_Customer_Review_2023

"The Amazon Product Review Dataset (2023) contains product reviews from Amazon customers. The dataset includes product information, review details, and metadata about the customers who left the reviews."

In [62]:
dataset = load_dataset("kevykibbz/Amazon_Customer_Review_2023")

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
dataset['train'][0]

{'rating': 5.0,
 'title': 'Crazy comfy!',
 'text': 'Not gonna lie- they are not much to look at. Lol. Luckily I’m one of those ppl that values things for function over looks & these function well so far. They are seriously one of the most comfortable pairs of socks I’ve owned in 5 decades.  I have not tried to wash them yet, so fingers crossed on that rn.  They feel very cushiony.  I wear them in my winter boots & just on my feet shoeless around my home.  I wish they came in more colors.  I’m one of those ppl that absolutely cannot stand toe seams on socks, but these have not bothered me at all.  I have super high arches so the only change I would make to the socks would be some compression there.  However, the socks fit perfectly as-is which really surprised me given my arches.  I just like having compression at my arches bc it feels good on them.  I wear a ladies 10-1/2 shoe- mens 8-1/2 and I bought the medium socks. They fit perfectly.  That’s never happened.  I had honestly expecte

In [5]:
target_asin = 'B07F3BDT8T'

In [6]:
def filter_by_asin_and_review_length(example):
    return (example["asin"] == target_asin) & (len(example['text']) >= 1000)

In [7]:
product_reviews = dataset['train'].filter(filter_by_asin_and_review_length)

In [8]:
len(product_reviews)

11

In [9]:
examples = product_reviews.shuffle(seed=SEED).select(range(8))

In [10]:
letters = []
for example in examples:
    letters.append(
        {
            "asin": example["asin"],
            "name": example.get("user_id", "Customer"),
            "text": example["text"],
            "rating": example["rating"]
        }
    )

In [11]:
letters

[{'asin': 'B07F3BDT8T',
  'name': 'AG5SAJDVPMJKM4CBN7B6NX3T2LCQ',
  'text': 'I really like these socks. They are soft, warm and thick. They are not itchy (I have sensitive skin and anything with wool usually really irritates my skin but these socks apparently have enough other material to counteract the itchiness of the wool content). They are not some shapeless bulky tube socks, though they’re thick so they do have more bulk than your typical sock. You can’t have true warmth without some bulk. These conform to your feet in all the right places. I noticed a few little threads that were loose but trimmed them off. I purchased them on sale and my only regret is that I didn’t buy more than one package while they were on sale and save myself some money. They are kind of pricey when not on sale but this is coming from an admitted penny pincher. I wish that they were organic, comprised of completely natural materials and locally made. (Let’s be honest, despite looking, I don’t think I have f

## Sentiment Analysis

In [12]:
from transformers import pipeline
from accelerate import Accelerator

In [14]:
sentiment_analyzer = pipeline(task='sentiment-analysis', max_length=512)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 4549.33it/s]


In [15]:
for letter in letters:
    scores = sentiment_analyzer(letter['text'])
    label = max(scores, key=lambda x: x['score'])
    letter['sentiment_label'] = label['label']
    letter['sentiment_score'] = label['score']

In [16]:
letters

[{'asin': 'B07F3BDT8T',
  'name': 'AG5SAJDVPMJKM4CBN7B6NX3T2LCQ',
  'text': 'I really like these socks. They are soft, warm and thick. They are not itchy (I have sensitive skin and anything with wool usually really irritates my skin but these socks apparently have enough other material to counteract the itchiness of the wool content). They are not some shapeless bulky tube socks, though they’re thick so they do have more bulk than your typical sock. You can’t have true warmth without some bulk. These conform to your feet in all the right places. I noticed a few little threads that were loose but trimmed them off. I purchased them on sale and my only regret is that I didn’t buy more than one package while they were on sale and save myself some money. They are kind of pricey when not on sale but this is coming from an admitted penny pincher. I wish that they were organic, comprised of completely natural materials and locally made. (Let’s be honest, despite looking, I don’t think I have f

Clearly the default model here is either not trained appropriately for this product review task, or perhaps the max_length restriction of 512 is having an impact. If we review the example for user AFPGBDJ44S7IAJEXKXY7HMLYMXTQ, we see a negative label even though they've rated the product 5. Reading the start of the review, much of the text is tangential to the specific product being reviewed.

In [17]:
letters[3]

{'asin': 'B07F3BDT8T',
 'name': 'AFPGBDJ44S7IAJEXKXY7HMLYMXTQ',
 'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks wer

### Sentiment Analysis: Another Model

Using: https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment

"This is a bert-base-multilingual-uncased model finetuned for sentiment analysis on product reviews in six languages: English, Dutch, German, French, Spanish, and Italian. It predicts the sentiment of the review as a number of stars (between 1 and 5)."

In [18]:
sentiment_analyzer = pipeline(
    task='sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    max_length=512,
    truncation=True
)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 8547.05it/s]


In [19]:
def stars_to_sentiment(label):
    stars = int(label.split()[0])
    return 'negative' if stars < 3 else 'neutral' if stars == 3 else 'positive'

In [20]:
for letter in letters:
    scores = sentiment_analyzer(letter['text'])
    label = max(scores, key=lambda x: x['score'])
    letter['predicted_rating'] = label['label']
    letter['sentiment_score'] = label['score']
    letter['sentiment_label'] = stars_to_sentiment(label['label'])

In [21]:
letters

[{'asin': 'B07F3BDT8T',
  'name': 'AG5SAJDVPMJKM4CBN7B6NX3T2LCQ',
  'text': 'I really like these socks. They are soft, warm and thick. They are not itchy (I have sensitive skin and anything with wool usually really irritates my skin but these socks apparently have enough other material to counteract the itchiness of the wool content). They are not some shapeless bulky tube socks, though they’re thick so they do have more bulk than your typical sock. You can’t have true warmth without some bulk. These conform to your feet in all the right places. I noticed a few little threads that were loose but trimmed them off. I purchased them on sale and my only regret is that I didn’t buy more than one package while they were on sale and save myself some money. They are kind of pricey when not on sale but this is coming from an admitted penny pincher. I wish that they were organic, comprised of completely natural materials and locally made. (Let’s be honest, despite looking, I don’t think I have f

So these "sentiment labels" now represent predicted product ratings and seem to be quite accurate. I'll use these predicted ratings to assign neutral to 3 stars, positive above, and negative below. The "sentiment score" can be used to identify the extremes.

## Summarise Letters

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [23]:
def make_summarizer(model_id="facebook/bart-large-cnn"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    def summarize(text, max_new_tokens=80):
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=1024,
        )
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_return_sequences=1,
            )
        summary = tokenizer.decode(out[0], skip_special_tokens=True)
        return summary

    return summarize

In [24]:
summarize = make_summarizer()

Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 511/511 [00:00<00:00, 8811.64it/s]


In [25]:
for letter in letters:
    letter['summary'] = summarize(letter['text'], max_new_tokens=256)

Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [26]:
letters[3]

{'asin': 'B07F3BDT8T',
 'name': 'AFPGBDJ44S7IAJEXKXY7HMLYMXTQ',
 'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks wer

## Question Answering

In [27]:
from transformers import AutoModelForQuestionAnswering

In [28]:
def make_question_answering_function(model_name='deepset/roberta-base-squad2'):
    model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def ask_question(question, context, max_length=384):
        inputs = tokenizer(
            question,
            context,
            max_length=max_length,
            truncation="only_second",
            return_tensors="pt",
        )
        with torch.no_grad():
            outputs = model(**inputs)
        
        answer_start_index = outputs.start_logits.argmax()
        answer_end_index = outputs.end_logits.argmax()

        predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index +1]
        answer = tokenizer.decode(predict_answer_tokens)
        
        return answer
        
    return ask_question

In [29]:
question_answering = make_question_answering_function()

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8413.55it/s]
RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
question_answering(
    question='What is the capital of France?',
    context='The capitals of Germany and France are Berlin and Paris respectively'
)

' Paris'

In [31]:
letters[3]

{'asin': 'B07F3BDT8T',
 'name': 'AFPGBDJ44S7IAJEXKXY7HMLYMXTQ',
 'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks wer

In [32]:
question_answering(
    question='What feature does the customer praise?',
    context=letters[3]['summary']
)

'<s>'

In [33]:
question_answering(
    question='What condition do the socks remain in?',
    context=letters[3]['summary']
)

' near new condition'

So this type of question-answering is quite good at fact-finding and extracting exact spans. They fall short of reconstructing open-ended intent though.

### QA: Text Generation Model

Using: https://huggingface.co/Qwen/Qwen3-8B

Docs are quite useful here: https://qwen.readthedocs.io/en/latest/inference/transformers.html

Downloaded locally with:
```hf download Qwen/Qwen3-8B --local-dir ./Qwen-8B```

In [34]:
import copy
import re

In [35]:
model_path = './Qwen-8B/'

In [36]:
qwen_qa_generator = pipeline(
    task='text-generation',
    model=model_path,
    dtype='auto',
    device_map='auto'
)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 399/399 [00:00<00:00, 451.00it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


In [37]:
messages = [
    {
        'role': 'system',
        'content': (
            'You are an assistant that reads customer product reviews.'
            'Your task is to answer questions about feedback the customer provides.'
            'You must answer in one short sentence'
        )
    },
    {
        'role': 'user',
        'content': (
            f"Here is a customer review summary:\n{letters[3]['summary']}\n"
            f"Question: What change, improvement, outcome, or product feature is the customer requesting or celebrating?"
        )
    },
]

In [38]:
output = qwen_qa_generator(messages, max_new_tokens=512)[0]['generated_text']

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [39]:
output

[{'role': 'system',
  'content': 'You are an assistant that reads customer product reviews.Your task is to answer questions about feedback the customer provides.You must answer in one short sentence'},
 {'role': 'user',
  'content': 'Here is a customer review summary:\nSock is high percentage wool, one of the highest available on the Internet to buy. They remain in near new condition in terms of softness, warmth, and durability. No holes or irritating seams, or weird shrinkages to be found. Hiking, farm work, gardening, camping, and walking around in them have not done a thing to them.\nQuestion: What change, improvement, outcome, or product feature is the customer requesting or celebrating?'},
 {'role': 'assistant',
  'content': "<think>\nOkay, let's see. The customer is talking about socks made of high percentage wool, which is one of the highest available online. They mention that the socks stay in near new condition in terms of softness, warmth, and durability. Also, there are no i

In [40]:
output[-1]['content']

"<think>\nOkay, let's see. The customer is talking about socks made of high percentage wool, which is one of the highest available online. They mention that the socks stay in near new condition in terms of softness, warmth, and durability. Also, there are no issues like holes, irritating seams, or weird shrinkage. They've used them for various activities like hiking, farming, gardening, camping, and walking, and the socks haven't been affected.\n\nThe question is asking what change, improvement, outcome, or product feature the customer is requesting or celebrating. So, the customer is celebrating the product's durability and quality. They want to highlight that the socks maintain their condition despite heavy use. The key points are the high wool content, durability, and absence of common issues. The customer isn't asking for anything new but is praising the existing features. So the answer should focus on the product's durability and quality retention over time.\n</think>\n\nThe custo

In [41]:
def parse_thinking_content(messages):
    messages = copy.deepcopy(messages)
    for message in messages:
        if message["role"] == "assistant" and (m := re.match(r"<think>\n(.+)</think>\n\n", message["content"], flags=re.DOTALL)):
            message["content"] = message["content"][len(m.group(0)):]
            if thinking_content := m.group(1).strip():
                message["reasoning_content"] = thinking_content
    return messages

In [42]:
parsed = parse_thinking_content(output)

In [43]:
parsed[-1]['content']

"The customer is celebrating the product's exceptional durability and quality retention, as the socks maintain their softness, warmth, and condition despite extensive use in various demanding activities."

Amazing, that's exactly what I want to see. Now to just encapsulate what we have to a helper function.

When looping through letters, I was hitting VRAM constraints so appending "/no_think" to the prompt.

In [44]:
def ask_qwen(generator, question, context, max_new_tokens):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are an assistant that reads customer product reviews.'
                'Your task is to answer questions about feedback the customer provides.'
                'You must answer in one short sentence'
            )
        },
        {
            'role': 'user',
            'content': (
                f"Here is a customer review summary:\n{context}\n"
                f"Question: {question}./no_think"
            )
        },
    ]

    output = generator(messages, max_new_tokens=max_new_tokens)[0]['generated_text']
    parsed = parse_thinking_content(output)

    return parsed[-1]['content']

In [45]:
QUESTION = 'What change, improvement, outcome, or product feature is the customer requesting or celebrating?'

In [46]:
for letter in letters:
    letter['feedback'] = ask_qwen(
        generator=qwen_qa_generator,
        question=QUESTION,
        context=letter['summary'],
        max_new_tokens=512
    )

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

In [47]:
letters[3]

{'asin': 'B07F3BDT8T',
 'name': 'AFPGBDJ44S7IAJEXKXY7HMLYMXTQ',
 'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks wer

## Text Generation: Letters of Reply

### Extremes

In [48]:
def stars_from_pred(pred):
    return int(pred.split()[0])

In [49]:
positives = [l for l in letters if stars_from_pred(l["predicted_rating"]) >= 3]
negatives = [l for l in letters if stars_from_pred(l["predicted_rating"]) < 3]

In [50]:
positives.sort(key = lambda l: (stars_from_pred(l['predicted_rating']), l['sentiment_score']), reverse=True)
negatives.sort(key = lambda l: (stars_from_pred(l['predicted_rating']), -l['sentiment_score']))

In [51]:
positives

[{'asin': 'B07F3BDT8T',
  'name': 'AFPGBDJ44S7IAJEXKXY7HMLYMXTQ',
  'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks 

In [52]:
negatives

[{'asin': 'B07F3BDT8T',
  'name': 'AGZIZIGXS2IZMRG3SM3YEEIONXIA',
  'text': 'One star because these socks wear out so quickly they simply aren\'t worth the money.  These would be 5 star socks if the heels didn\'t wear out so quickly.  Press the tips of your thumb and index finger together and you\'ve got the size of the holes in four out of six socks, purchased 51 days ago--and not worn daily.  My husband likes the socks because they are warm, cushion his feet well, and have more size choices than most socks  He wears a 9.5 shoe and the medium fits him well with no bunching up of extra material in the toes.  Working outdoors and walking a lot means he takes good care of his feet and only wears good leather boots or pull-on lined "rubber" boots.  He\'s like The Princess and the Pea about anything irritating or abrasive against the soles of his feet, so there is nothin in his footwear to blame, the flaw is in the socks.  It\'s too bad because the toes and the ball of the foot, basically 

In [53]:
extremes = [lst[0] for lst in (positives, negatives) if lst]

In [54]:
extremes

[{'asin': 'B07F3BDT8T',
  'name': 'AFPGBDJ44S7IAJEXKXY7HMLYMXTQ',
  'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks 

### Generated Replies

In [55]:
def qwen_reply(generator, customer, summary, feedback, sentiment, max_new_tokens):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a customer support manager responding to customer product reviews and feedback.'
                'Your task is to write a short, empathetic reply to each customer based on their review and sentiment.'
                f"You must always start with Dear {customer},' and end appropriately with 'Best regards, \n Alvada Customer Support'."
                "Be specific about the product and the customer's feedback."
            )
        },
        {
            'role': 'user',
            'content': (
                f"Customer Name: {customer}\n"
                f"Summary of the review: {summary}."
                f"The customer's feedback: {feedback}."
                f"The customer's sentiment: {sentiment}."
                'Write a short email reply to the customer./no_think'
            )
        },
    ]

    output = generator(messages, max_new_tokens=max_new_tokens)[0]['generated_text']
    parsed = parse_thinking_content(output)

    return parsed[-1]['content']

In [56]:
for letter in extremes:
    letter['reply'] = qwen_reply(
        generator=qwen_qa_generator,
        customer=letter["name"],
        summary=letter["summary"],
        feedback=letter["feedback"],
        sentiment=letter["sentiment_label"],
        max_new_tokens=512
    )

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [57]:
extremes

[{'asin': 'B07F3BDT8T',
  'name': 'AFPGBDJ44S7IAJEXKXY7HMLYMXTQ',
  'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks 

## Generate Report

In [58]:
from docx import Document
from docx.shared import Pt, RGBColor
from datetime import datetime

In [60]:
def create_manager_report(
    positive_letters,
    negative_letters,
    extremes,
    output_path="customer_feedback_report.docx",
):
    doc = Document()
    
    # Title
    doc.add_heading("Customer Feedback Report", level=0)
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph("Product: https://www.amazon.com/Alvada-Merino-Hiking-Thermal-Winter/dp/B07F3BDT8T?th=1&psc=1")
    
    # Executive Summary
    doc.add_heading("Executive Summary", level=1)
    doc.add_paragraph(
        f"This report analyzes {len(positive_letters) + len(negative_letters)} customer reviews "
        f"for the B07F3BDT8T product. {len(positive_letters)} reviews were positive (rating ≥ 3 stars), "
        f"and {len(negative_letters)} were negative (rating < 3 stars)."
    )
    
    # Positive Responses Section
    doc.add_heading("Neutral & Positive Responses", level=1)
    doc.add_paragraph(f"Total positive reviews: {len(positive_letters)}")
    for i, letter in enumerate(positive_letters, 1):
        p = doc.add_paragraph()
        p.add_run(f"{i}. {letter['name']} (Rating: {letter['predicted_rating']}):").bold = True
        p.add_run(letter['feedback'])
    
    # Negative Responses Section
    doc.add_heading("Negative Responses", level=1)
    doc.add_paragraph(f"Total negative reviews: {len(negative_letters)}")
    for i, letter in enumerate(negative_letters, 1):
        p = doc.add_paragraph()
        p.add_run(f"{i}. {letter['name']} (Rating: {letter['predicted_rating']}):").bold = True
        p.add_run(letter['feedback'])
    
    # Most Extreme Sentiments
    doc.add_heading("Most Extreme Examples", level=1)
    
    most_positive = extremes[0]
    most_negative = extremes[1]
    
    # Most Positive
    doc.add_heading("Most Positive: Customer Intent", level=2)
    doc.add_paragraph(f"Customer: {most_positive['name']}")
    doc.add_paragraph(f"Rating: {most_positive['predicted_rating']}")
    doc.add_paragraph(f"Summary: {most_positive['summary']}")
    doc.add_paragraph(f"Feedback: {most_positive['feedback']}")
    
    # Most Negative
    doc.add_heading("Most Negative: Customer Intent", level=2)
    doc.add_paragraph(f"Customer: {most_negative['name']}")
    doc.add_paragraph(f"Rating: {most_negative['predicted_rating']}")
    doc.add_paragraph(f"Summary: {most_negative['summary']}")
    doc.add_paragraph(f"Feedback: {most_negative['feedback']}")
    
    # Automated Replies
    doc.add_heading("Automated Replies to Extreme Examples", level=1)
    
    doc.add_heading(f"Reply to {most_positive['name']}", level=2)
    doc.add_paragraph(most_positive['reply'])
    
    doc.add_heading(f"Reply to {most_negative['name']}", level=2)
    doc.add_paragraph(most_negative['reply'])
    
    # Design Choices Appendix
    doc.add_heading("Design Choices", level=1)
    choices = [
        'Generated input "Letters": Amazon Product Review Dataset (2023). https://huggingface.co/datasets/kevykibbz/Amazon_Customer_Review_2023', 
        "Sentiment Model: nlptown/bert-base-multilingual-uncased-sentiment (trained on Amazon reviews). https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment",
        "Sentiment mapping: 1–2 stars → negative, 3 stars → neutral, 4–5 stars → positive",
        "Summarization Model: facebook/bart-large-cnn for abstractive summaries. https://huggingface.co/facebook/bart-large-cnn",
        "Feedback Extraction: Qwen3-8B local model with structured question-answering prompts. https://huggingface.co/Qwen/Qwen3-8B",
        "Reply Generation: Same Qwen3-8B local model with customer support assistant prompts.",
        "Extremes Selection: Ordered by (predicted_rating, sentiment_score) to find best/worst",
    ]
    for choice in choices:
        doc.add_paragraph(choice, style="List Bullet")
    
    doc.save(output_path)
    print(f"✓ Report saved to {output_path}")


In [61]:
create_manager_report(
    positive_letters=positives,
    negative_letters=negatives,
    extremes=extremes,
    output_path='customer_feedback_report.docx'
)

✓ Report saved to customer_feedback_report.docx
